In [1]:
import gc
from pathlib import Path

import numpy as np
import polars as pl
import polars.selectors as cs

# Configure Polars to print everything
pl.Config.set_tbl_rows(30)  # -1 means display all rows
pl.Config.set_tbl_cols(-1)  # -1 means display all columns


polars.config.Config

In [2]:
# I/O Constants
INPUT_BASE_PATH = Path("/kaggle/input/competitions/m5-forecasting-accuracy")
OUTPUT_BASE_PATH = Path("/kaggle/working")

# Data constants
ID_COLS = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
N_EVENTS = 2
MAX_TRAIN_TIMESTAMP = 1941

# Feature engineering constants
SALES_LAGS = [1, 2, 3, 4, 7, 14]
ROLLING_SALES_WINDOWS = [2, 3, 4, 7, 14]


## LOAD DATA

In [3]:
# Calendar data
calendar_schema_overrides = {
    "wm_yr_wk": pl.Int16,
    **{date_type: pl.Int16 for date_type in ["wday", "month", "year"]},
    **{f"snap_{state}": pl.Int16 for state in ["CA", "TX", "WI"]}
}
raw_calendar_df = pl.read_csv(
    f"{INPUT_BASE_PATH}/calendar.csv",
    try_parse_dates=True,
    schema_overrides=calendar_schema_overrides,
)

# Sell prices
sell_price_schema_overrides = {
    "wm_yr_wk": pl.Int16,
    "sell_price": pl.Float32,
}
raw_sell_price_df = pl.read_csv(
    f"{INPUT_BASE_PATH}/sell_prices.csv",
    schema_overrides=sell_price_schema_overrides
)

# Sales data
sales_schema_overrides = {
    f"d_{t}": pl.Int16
    for t in range(1, MAX_TRAIN_TIMESTAMP + 1)
}
raw_sales_df = pl.read_csv(
    INPUT_BASE_PATH / "sales_train_evaluation.csv",
    schema_overrides=sales_schema_overrides,
)

In [4]:
# Merge all raw dataframes into feature df.

feature_df = (
    raw_sales_df
    # Cast keys to categorical on the small wide frame, before unpivoting to ~59M rows
    .with_columns(pl.col(ID_COLS).cast(pl.Categorical))
    .unpivot(
        index=ID_COLS,
        variable_name="d",
        value_name="sales",
    )
    .with_columns(
        pl.col("sales").cast(pl.Int16),
        d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16),
    )
    # Drop string d column, no longer needed.
    # d_index is the key from here on
    .drop("d")
    .join(
        # Initially only joining on a subset of columns
        other=raw_calendar_df.select(
            "wm_yr_wk",
            d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16),
        ),
        on="d_index",
        how="left",
    )
    .join(
        other=raw_sell_price_df.with_columns(pl.col("store_id", "item_id").cast(pl.Categorical)),
        on=["store_id", "item_id", "wm_yr_wk"],
        how="left",
    )
)

In [5]:
# Raw inputs are no longer needed, free them from memory
del raw_sales_df, raw_sell_price_df
gc.collect()

50

In [6]:
# Remove rows where price is null
feature_df = feature_df.filter(pl.col("sell_price").is_not_null())

# Check no null prices
assert not feature_df["sell_price"].is_null().any()

In [7]:
# Check no duplicates

assert feature_df.select(pl.col("id"), pl.col("d_index")).is_duplicated().sum() == 0

In [8]:
# Check that timeseries are continuous

id_series_lengths = (
    feature_df
    .group_by("id")
    .agg(
        d_index_min=pl.col("d_index").min(),
        d_index_max=pl.col("d_index").max(),
    )
    .with_columns(min_max_diff=pl.col("d_index_max") - pl.col("d_index_min"))
    .join(
        other=feature_df.group_by("id").len(name="series_length"),
        on="id",
        how="inner",
        validate="1:1"
    )
)

assert id_series_lengths["min_max_diff"].eq(id_series_lengths["series_length"] - 1).all()

## LAGGED SALES FEATURES

In [9]:
# Add columns directly on main features_df, to avoid expensive joins later on

feature_df = feature_df.with_columns(
    [
        pl.col("sales")
        .shift(lag)
        .over("id", order_by="d_index")
        .alias(f"sales_lagged_{lag}")
        for lag in SALES_LAGS
    ]
)


(
    feature_df
    .select(["id", "d_index", "sales", *[f"sales_lagged_{lag}" for lag in SALES_LAGS]])
    .sort(by=["id", "d_index"])
    .head(max(SALES_LAGS) + 2)
)

id,d_index,sales,sales_lagged_1,sales_lagged_2,sales_lagged_3,sales_lagged_4,sales_lagged_7,sales_lagged_14
cat,i16,i16,i16,i16,i16,i16,i16,i16
"""FOODS_1_001_CA_1_evaluation""",1,3,null,null,null,null,null,null
"""FOODS_1_001_CA_1_evaluation""",2,0,3,null,null,null,null,null
"""FOODS_1_001_CA_1_evaluation""",3,0,0,3,null,null,null,null
"""FOODS_1_001_CA_1_evaluation""",4,1,0,0,3,null,null,null
"""FOODS_1_001_CA_1_evaluation""",5,4,1,0,0,3,null,null
"""FOODS_1_001_CA_1_evaluation""",6,2,4,1,0,0,null,null
"""FOODS_1_001_CA_1_evaluation""",7,0,2,4,1,0,null,null
"""FOODS_1_001_CA_1_evaluation""",8,2,0,2,4,1,3,null
"""FOODS_1_001_CA_1_evaluation""",9,0,2,0,2,4,0,null


## ROLLING SALES FEATURES

In [10]:
# Add columns directly on main features_df, to avoid expensive joins later on

feature_df = feature_df.with_columns(
    [
        pl.col("sales")
        .shift(1)
        .rolling_mean(window_size=window, min_samples=window)
        .over("id", order_by="d_index")
        .alias(f"sales_rolling_mean_{window}")
        .cast(pl.Float32)
        for window in ROLLING_SALES_WINDOWS
    ]
)


(
    feature_df
    .select(["id", "d_index", "sales", *[f"sales_rolling_mean_{w}" for w in ROLLING_SALES_WINDOWS]])
    .sort(by=["id", "d_index"])
    .head(max(SALES_LAGS) + 2)
)


id,d_index,sales,sales_rolling_mean_2,sales_rolling_mean_3,sales_rolling_mean_4,sales_rolling_mean_7,sales_rolling_mean_14
cat,i16,i16,f32,f32,f32,f32,f32
"""FOODS_1_001_CA_1_evaluation""",1,3,null,null,null,null,null
"""FOODS_1_001_CA_1_evaluation""",2,0,null,null,null,null,null
"""FOODS_1_001_CA_1_evaluation""",3,0,1.5,null,null,null,null
"""FOODS_1_001_CA_1_evaluation""",4,1,0.0,1.0,null,null,null
"""FOODS_1_001_CA_1_evaluation""",5,4,0.5,0.333333,1.0,null,null
"""FOODS_1_001_CA_1_evaluation""",6,2,2.5,1.666667,1.25,null,null
"""FOODS_1_001_CA_1_evaluation""",7,0,3.0,2.333333,1.75,null,null
"""FOODS_1_001_CA_1_evaluation""",8,2,1.0,2.0,1.75,1.428571,null
"""FOODS_1_001_CA_1_evaluation""",9,0,1.0,1.333333,2.0,1.285714,null


## LAGGED PRICE FEATURES

### TODO

## ROLLING PRICE FEATURES

### TODO

## EVENT FEATURES

In [11]:
# TODO: Cast event types and names to categorical dtypes.

event_features_df = (
    raw_calendar_df
    .select(["wm_yr_wk", "d", "date", "event_name_1", "event_type_1", "event_name_2", "event_type_2"])
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16))
    .with_columns(
        pl.when(pl.col(f"event_name_{i}").is_not_null())
        .then(pl.col("d_index"))
        .otherwise(None)
        .alias(f"event_index_{i}")
        for i in (1, 2)
    )
    # Days until next / since last event
    .with_columns(
        # Days since last event
        *[
            (pl.col("d_index") - pl.col(f"event_index_{i}").forward_fill())
            .alias(f"days_since_event_{i}")
            for i in (1, 2)
        ],
        # Last event type
        *[
            (pl.col(f"event_type_{i}").forward_fill()
            .alias(f"last_event_type_{i}"))
            for i in range(1, N_EVENTS + 1)
        ],
        # Days until next event
        *[
            (pl.col(f"event_index_{i}").backward_fill() - pl.col("d_index"))
            .alias(f"days_until_event_{i}")
            for i in (1, 2)
        ],
        # Next event type
        *[
            (pl.col(f"event_type_{i}").backward_fill()
            .alias(f"next_event_type_{i}"))
            for i in range(1, N_EVENTS + 1)
        ],
    )
    .with_columns(
        # Clip days since last event and event type to 7 days.
        *[
            pl.when(pl.col(f"days_since_event_{i}") <= 7)
            .then(pl.col(f"days_since_event_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.when(pl.col(f"days_since_event_{i}") <= 7).
            then(pl.col(f"last_event_type_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
        # Clip days until next event and event type to 7 days.
        *[              
            pl.when(pl.col(f"days_until_event_{i}") <= 7)
            .then(pl.col(f"days_until_event_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.when(pl.col(f"days_until_event_{i}") <= 7)
            .then(pl.col(f"next_event_type_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ]
    )
)

event_features_df.head()

wm_yr_wk,d,date,event_name_1,event_type_1,event_name_2,event_type_2,d_index,event_index_1,event_index_2,days_since_event_1,days_since_event_2,last_event_type_1,last_event_type_2,days_until_event_1,days_until_event_2,next_event_type_1,next_event_type_2
i16,str,date,str,str,str,str,i16,i16,i16,i16,i16,str,str,i16,i16,str,str
11101,"""d_1""",2011-01-29,null,null,null,null,1,null,null,null,null,null,null,null,null,null,null
11101,"""d_2""",2011-01-30,null,null,null,null,2,null,null,null,null,null,null,7,null,"""Sporting""",null
11101,"""d_3""",2011-01-31,null,null,null,null,3,null,null,null,null,null,null,6,null,"""Sporting""",null
11101,"""d_4""",2011-02-01,null,null,null,null,4,null,null,null,null,null,null,5,null,"""Sporting""",null
11101,"""d_5""",2011-02-02,null,null,null,null,5,null,null,null,null,null,null,4,null,"""Sporting""",null


## CALENDAR FEATURS

In [12]:
calendar_features_df = (
    raw_calendar_df
    .select(["wm_yr_wk", "d", "date", "wday", "month", "year"])
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16))
    .with_columns(
        sin_wday=(2 * np.pi * pl.col("wday") / 7).sin().cast(pl.Float32),
        cos_wday=(2 * np.pi * pl.col("wday") / 7).cos().cast(pl.Float32),
        sin_month=(2 * np.pi * pl.col("month") / 12).sin().cast(pl.Float32),
        cos_month=(2 * np.pi * pl.col("month") / 12).cos().cast(pl.Float32),
        is_weekend=(pl.col("wday") <= 2).cast(pl.Float32),
    )
)

calendar_features_df.head()

wm_yr_wk,d,date,wday,month,year,d_index,sin_wday,cos_wday,sin_month,cos_month,is_weekend
i16,str,date,i16,i16,i16,i16,f32,f32,f32,f32,f32
11101,"""d_1""",2011-01-29,1,1,2011,1,0.781832,0.62349,0.5,0.866025,0.0
11101,"""d_2""",2011-01-30,2,1,2011,2,0.974928,-0.222521,0.5,0.866025,0.0
11101,"""d_3""",2011-01-31,3,1,2011,3,0.433884,-0.900969,0.5,0.866025,0.0
11101,"""d_4""",2011-02-01,4,2,2011,4,-0.433884,-0.900969,0.866025,0.5,0.0
11101,"""d_5""",2011-02-02,5,2,2011,5,-0.974928,-0.222521,0.866025,0.5,0.0


## ELECTION FEATURES

### TODO

## MERGE ALL FEATURES

In [13]:
# Merge small calendar and event features df first, drop redunant columns and 
# cast to categorical before merging onto large features df.
events_and_calendar_features_df = (
    calendar_features_df.drop("d", "date")
    .join(
        other=event_features_df.drop("d", "date"),
        on=["wm_yr_wk", "d_index"],
        how="inner",
        validate="1:1",
    )
    .with_columns(cs.string().cast(pl.Categorical))
)

final_features_df = feature_df.join(
    other=events_and_calendar_features_df,
    on=["wm_yr_wk", "d_index"],
    how="inner",
    validate="m:1",
)

In [14]:
# Write to output
final_features_df.write_parquet(OUTPUT_BASE_PATH / "m5-acc-features-df.pq")